In [ ]:
import importlib
import sys

import pandas as pd
import numpy as np

sys.path.append("../src")

import data_splitting as split
import eda_utils as eda
import feature_engineering as fe
import modeling as mod
import preprocessing as prep
import constants as const
import visualization as visual

importlib.reload(eda)
importlib.reload(fe)
importlib.reload(mod)
importlib.reload(prep)
importlib.reload(const)
importlib.reload(visual)

In [ ]:
%load_ext autoreload
%autoreload 2

<h1 style="
    background-color: #d0ebff;
    color: #1a1a1a;
    display: inline-block;
    padding: 10px 18px;
    border-radius: 10px;
    font-size: 32px;
">
Feature Engineering Experiments
</h1>


Este notebook reúne pruebas de feature engineering orientadas a mejorar la predicción de precios. En particular, se evalúa si la columna `Versión` aporta información útil sobre el nivel de equipamiento del vehículo. Se busca la mejor identificación de autos de alta gama.

In [ ]:
dataset_processed = pd.read_csv("../data/dataset_pp.csv")

X_train = pd.read_csv("../data/processed/splitted/X_train.csv")
X_val = pd.read_csv("../data/processed/splitted/X_val.csv")

y_train = pd.read_csv("../data/processed/splitted/y_train.csv").squeeze()
y_val = pd.read_csv("../data/processed/splitted/y_val.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Analysis
</h3>


La variable `Versión` contiene información sobre equipamiento, tracción y estilo de la publicación. En lugar de usarla directamente como texto libre, se busca extraer señales más simples: un nivel ordinal de versión y algunos indicadores binarios.

Antes de crear las variables, revisamos cómo varían las versiones dentro de cada combinación de `Marca` y `Modelo`.


In [ ]:
version_summary = (
    dataset_processed
    .groupby(["Marca", "Modelo", "Versión"])["Precio"]
    .agg(
        count="count",
        median_price="median",
        mean_price="mean",
    )
    .reset_index()
    .sort_values(["Marca", "Modelo", "count"], ascending=[True, True, False])
)

version_summary.head(50)


Se observa clarament ecomo para una misma marca y modelo hay un rango muy altos de precios

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Version Feature Definitions
</h3>


Se definen patrones de texto para identificar versiones base, medias, altas y premium. Como la clasificación no es perfecta, también se crea una variable `Version_Tier_Unknown` para marcar los casos donde no se detectó ninguna palabra clave.

Además del tier ordinal, se agregan indicadores binarios para señales que pueden afectar el precio pero no representan exactamente un nivel de equipamiento, como tracción `4x4`, versiones deportivas o extras.


In [ ]:
PREMIUM_VERSION_PATTERNS = [
    r"\bpremium\b",
    r"\bplatinum\b",
    r"\bsummit\b",
    r"\boverland\b",
    r"\bhigh country\b",
    r"\bhigh altitude\b",
    r"\bluxury\b",
    r"\btouring\b",
    r"\bamg\b",
    r"\bm sport\b",
    r"\bm performance\b",
    r"\bs-?line\b",
    r"\br-?line\b",
    r"\braptor\b",
    r"\bsahara\b",
    r"\brubicon\b",
]

HIGH_VERSION_PATTERNS = [
    r"\blimited\b",
    r"\bhighline\b",
    r"\btitanium\b",
    r"\bpremier\b",
    r"\bexclusive\b",
    r"\bfeline\b",
    r"\bshine\b",
    r"\btrailhawk\b",
    r"\bsrx\b",
    r"\bseg\b",
    r"\bxle\b",
    r"\bex-?l\b",
    r"\biconic\b",
    r"\bintens\b",
    r"\bimpetus\b",
    r"\baudace\b",
    r"\bxline\b",
    r"\bsx\b",
    r"\bltz\+?\b",
    r"\bserie[- ]?s\b",
]

MID_VERSION_PATTERNS = [
    r"\bcomfortline\b",
    r"\ballure\b",
    r"\badvance\b",
    r"\bxei\b",
    r"\bxls\b",
    r"\bsrv\b",
    r"\bex\b",
    r"\bgls\b",
    r"\bsel\b",
    r"\blt\b",
    r"\blongitude\b",
    r"\bprivilege\b",
    r"\bdynamique\b",
    r"\bzen\b",
    r"\bfeel\b",
    r"\bstyle\b",
    r"\bfreestyle\b",
    r"\blife\b",
    r"\bse\b",
    r"\bdrive\b",
]

BASE_VERSION_PATTERNS = [
    r"\bbase\b",
    r"\bentry\b",
    r"\btrendline\b",
    r"\bsense\b",
    r"\bactive\b",
    r"\bxli\b",
    r"\bls\b",
    r"\blx\b",
    r"\bxl\b",
    r"\b1lt\b",
    r"\bexpression\b",
    r"\bgl\b",
]

VERSION_4X4_PATTERNS = [
    r"\b4x4\b",
    r"\b4wd\b",
    r"\bawd\b",
    r"\b4motion\b",
    r"\bxdrive\b",
    r"\bquattro\b",
    r"\b4matic\b",
]

VERSION_SPORT_PATTERNS = [
    r"\bsport\b",
    r"\bsportline\b",
    r"\bgt\b",
    r"\bgt[- ]?line\b",
    r"\brs\b",
    r"\babarth\b",
    r"\bgr[- ]?s\b",
    r"\btrailhawk\b",
]

VERSION_EXTRA_PATTERNS = [
    r"\bplus\b",
    r"\bpack\b",
    r"\bpk\b",
    r"\bfull\b",
    r"\btop\b",
    r"\btecho\b",
    r"\bcuero\b",
    r"\bnav\b",
    r"\bbitono\b",
    r"\bbi tono\b",
]

VERSION_TIER_PATTERNS = {
    3: PREMIUM_VERSION_PATTERNS,
    2: HIGH_VERSION_PATTERNS,
    1: MID_VERSION_PATTERNS,
    0: BASE_VERSION_PATTERNS,
}

VERSION_BINARY_PATTERNS = {
    "4x4": VERSION_4X4_PATTERNS,
    "Sport": VERSION_SPORT_PATTERNS,
    "Extra": VERSION_EXTRA_PATTERNS,
}


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Creation
</h3>


In [ ]:
X_train_fe = fe.add_text_pattern_features(
    X_train,
    source_col="Versión",
    tier_patterns=VERSION_TIER_PATTERNS,
    binary_patterns=VERSION_BINARY_PATTERNS,
    prefix="Version",
    default_tier=1,
    drop_source_col=True,
)

X_val_fe = fe.add_text_pattern_features(
    X_val,
    source_col="Versión",
    tier_patterns=VERSION_TIER_PATTERNS,
    binary_patterns=VERSION_BINARY_PATTERNS,
    prefix="Version",
    default_tier=1,
    drop_source_col=True,
)

X_train_fe = prep.drop_irrelevant_columns(
    X_train_fe,
    columns_to_drop=["Título", "Descripción"],
)

X_val_fe = prep.drop_irrelevant_columns(
    X_val_fe,
    columns_to_drop=["Título", "Descripción"],
)

X_train_fe.head()


In [ ]:
# Encoding the dataset
categorical_cols_to_encode = [
    "Marca",
    "Modelo",
    "Color",
    "Tipo de vendedor",
    "Tipo de combustible",
    "Transmisión",
]

X_train_encoded, categories_map = prep.one_hot_encoding(
    X_train_fe,
    categorical_cols=categorical_cols_to_encode,
    train=True,
    binary_missing_cols=["Con cámara de retroceso"],
)

X_val_encoded = prep.one_hot_encoding(
    X_val_fe,
    categorical_cols=categorical_cols_to_encode,
    train=False,
    categories_map=categories_map,
    binary_missing_cols=["Con cámara de retroceso"],
)

print(f"X_train_encoded shape: {X_train_encoded.shape}")
print(f"X_val_encoded shape: {X_val_encoded.shape}")


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Model Training
</h3>


Para entrenar este experimento, se utiliza un modelo XGBoost con el precio transformado a escala logarítmica. Se comparan dos estrategias:

- **Modelo único:** se entrena un solo XGBoost con todas las observaciones.
- **Modelo segmentado:** se separa el dataset en vehículos de alta gama y vehículos no alta gama, y se entrena un XGBoost distinto para cada grupo.

La idea es evaluar si separar muestras con rangos de precio y comportamiento potencialmente distintos ayuda a reducir el ruido y mejorar la performance de validación.


In [ ]:
xgboost_single_model, xgboost_single_metrics, xgboost_single_predictions = mod.train_xgboost(
    X_train_encoded,
    y_train,
    X_val_encoded,
    y_val,
    use_log_target=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
)

xgboost_single_metrics


In [ ]:
visual.plot_regression_metrics(
    xgboost_single_metrics,
    model_name="XGBoost Log - Single Model",

)


#### High-end split

Para la segunda estrategia, la separación se realiza desde el dataset preprocesado completo. Primero se divide `dataset_processed` en dos subconjuntos según si la marca pertenece al grupo de alta gama o no. Luego, dentro de cada subconjunto, se vuelve a hacer el split train/validation y recién después se aplica el one-hot encoding por separado.

De esta forma, cada modelo se entrena con un universo de categorías propio: el modelo de alta gama solo aprende categorías presentes en autos de alta gama, y el modelo del resto solo aprende categorías presentes en autos no alta gama.


In [ ]:
TARGET = "Precio"

dataset_high_end_mask = dataset_processed["Marca"].isin(const.PREMIUM_BRANDS)

dataset_high_end = dataset_processed.loc[dataset_high_end_mask].copy()
dataset_regular = dataset_processed.loc[~dataset_high_end_mask].copy()

X_high_end = dataset_high_end.drop(columns=[TARGET])
y_high_end = dataset_high_end[TARGET]

X_regular = dataset_regular.drop(columns=[TARGET])
y_regular = dataset_regular[TARGET]

(X_train_high_end_raw, y_train_high_end), (X_val_high_end_raw, y_val_high_end) = split.train_val_split_stratified(
    X_high_end,
    y_high_end,
    stratify_by=X_high_end["Marca"],
    train_size=0.80,
    random_state=42,
)

(X_train_regular_raw, y_train_regular), (X_val_regular_raw, y_val_regular) = split.train_val_split_stratified(
    X_regular,
    y_regular,
    stratify_by=X_regular["Marca"],
    train_size=0.80,
    random_state=42,
)

version_feature_kwargs = {
    "source_col": "Versión",
    "tier_patterns": VERSION_TIER_PATTERNS,
    "binary_patterns": VERSION_BINARY_PATTERNS,
    "prefix": "Version",
    "default_tier": 1,
    "drop_source_col": True,
    "columns_to_drop": ["Título", "Descripción"],
}

X_train_high_end_segment, X_val_high_end_segment = fe.add_text_pattern_features_to_split(
    X_train_high_end_raw,
    X_val_high_end_raw,
    **version_feature_kwargs,
)

X_train_regular_segment, X_val_regular_segment = fe.add_text_pattern_features_to_split(
    X_train_regular_raw,
    X_val_regular_raw,
    **version_feature_kwargs,
)

split_size_summary = pd.DataFrame([
    {
        "split": "train",
        "high_end_rows": len(X_train_high_end_segment),
        "non_high_end_rows": len(X_train_regular_segment),
    },
    {
        "split": "validation",
        "high_end_rows": len(X_val_high_end_segment),
        "non_high_end_rows": len(X_val_regular_segment),
    },
])

split_size_summary.style.hide(axis="index")


In [ ]:
XGBOOST_PARAMS = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
}


In [ ]:
X_train_high_end_encoded, high_end_categories_map = prep.one_hot_encoding(
    X_train_high_end_segment,
    categorical_cols=categorical_cols_to_encode,
    train=True,
    binary_missing_cols=["Con cámara de retroceso"],
)

X_val_high_end_encoded = prep.one_hot_encoding(
    X_val_high_end_segment,
    categorical_cols=categorical_cols_to_encode,
    train=False,
    categories_map=high_end_categories_map,
    binary_missing_cols=["Con cámara de retroceso"],
)

X_train_regular_encoded, regular_categories_map = prep.one_hot_encoding(
    X_train_regular_segment,
    categorical_cols=categorical_cols_to_encode,
    train=True,
    binary_missing_cols=["Con cámara de retroceso"],
)

X_val_regular_encoded = prep.one_hot_encoding(
    X_val_regular_segment,
    categorical_cols=categorical_cols_to_encode,
    train=False,
    categories_map=regular_categories_map,
    binary_missing_cols=["Con cámara de retroceso"],
)

pd.DataFrame([
    {
        "segment": "high_end",
        "train_shape": X_train_high_end_encoded.shape,
        "validation_shape": X_val_high_end_encoded.shape,
    },
    {
        "segment": "non_high_end",
        "train_shape": X_train_regular_encoded.shape,
        "validation_shape": X_val_regular_encoded.shape,
    },
])


#### Single model with high-end feature

Dado que el split por alta gama deja pocas observaciones en ese segmento, se prueba una alternativa extra más estable, en caso de que el split no funcione: mantener un único modelo para todo el dataset y agregar una variable binaria que indique si la marca pertenece al grupo de alta gama. De esta forma, el modelo conserva toda la información disponible y, al mismo tiempo, puede ajustar diferencias sistemáticas entre autos premium y no premium.


In [ ]:
single_model_high_end_mask_train = X_train_fe["Marca"].isin(const.PREMIUM_BRANDS)
single_model_high_end_mask_val = X_val_fe["Marca"].isin(const.PREMIUM_BRANDS)

X_train_high_end_feature = X_train_encoded.copy()
X_val_high_end_feature = X_val_encoded.copy()

X_train_high_end_feature["Es_Alta_Gama"] = single_model_high_end_mask_train.astype(int)
X_val_high_end_feature["Es_Alta_Gama"] = single_model_high_end_mask_val.astype(int)

xgboost_high_end_feature_model, xgboost_high_end_feature_metrics, xgboost_high_end_feature_predictions = mod.train_xgboost(
    X_train_high_end_feature,
    y_train,
    X_val_high_end_feature,
    y_val,
    use_log_target=True,
    **XGBOOST_PARAMS,
)

xgboost_high_end_feature_metrics


In [ ]:
high_end_feature_comparison = pd.concat(
    [
        xgboost_single_metrics.assign(model="single_model"),
        xgboost_high_end_feature_metrics.assign(model="single_model_with_high_end_feature"),
    ],
    ignore_index=True,
)

visual.plot_regression_metrics(
    high_end_feature_comparison,
    model_name="XGBoost Log - Single Model vs High-End Feature",
    label_col="model",
)

high_end_feature_comparison


#### High-end model

Primero se entrena un XGBoost usando solamente las observaciones de marcas consideradas de alta gama.


In [ ]:
xgboost_high_end_model, xgboost_high_end_metrics, xgboost_high_end_predictions = mod.train_xgboost(
    X_train_high_end_encoded,
    y_train_high_end,
    X_val_high_end_encoded,
    y_val_high_end,
    use_log_target=True,
    **XGBOOST_PARAMS,
)

xgboost_high_end_metrics


#### "Rest" model

Luego se entrena otro XGBoost independiente con el resto de las observaciones. De esta forma, el modelo segmentado queda compuesto por dos modelos distintos.


In [ ]:
xgboost_regular_model, xgboost_regular_metrics, xgboost_regular_predictions = mod.train_xgboost(
    X_train_regular_encoded,
    y_train_regular,
    X_val_regular_encoded,
    y_val_regular,
    use_log_target=True,
    **XGBOOST_PARAMS,
)

xgboost_regular_metrics


In [ ]:
xgboost_segment_metrics = pd.concat(
    [
        xgboost_high_end_metrics.assign(segment="high_end"),
        xgboost_regular_metrics.assign(segment="non_high_end"),
    ],
    ignore_index=True,
)

visual.plot_regression_metrics(
    xgboost_segment_metrics,
    model_name="XGBoost Log - Separate Segment Models",
    label_col="segment",
)


##### Analizando resultados

Claramente se observa como, si bien el modelo resto mejora mucho la aperformance, el modelo de alta gama empeora mucho, veamos que estpá ocurrinedo

In [ ]:
pd.DataFrame([
    {
        "segment": "high_end",
        "train_count": len(X_train_high_end_segment),
        "val_count": len(X_val_high_end_segment),
    },
    {
        "segment": "non_high_end",
        "train_count": len(X_train_regular_segment),
        "val_count": len(X_val_regular_segment),
    },
])


In [ ]:
high_end_price_summary = (
    X_train_high_end_raw
    .assign(Precio=y_train_high_end)
    .groupby(["Marca", "Modelo"])["Precio"]
    .agg(count="count", median="median", mean="mean", min="min", max="max")
    .sort_values("max", ascending=False)
)

high_end_price_summary.head(20)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Feature Diagnostics
</h3>


Luego de entrenar, revisamos si las variables creadas tienen cobertura suficiente y si el modelo efectivamente las utiliza. Esto ayuda a decidir si vale la pena conservarlas en el pipeline final.


In [ ]:
version_feature_cols = [
    "Version_Tier",
    "Version_Tier_Unknown",
    "Version_4x4",
    "Version_Sport",
    "Version_Extra",
]

X_val_fe[version_feature_cols].mean().sort_values(ascending=False)


In [ ]:
diagnostic_data = X_train_fe.copy()
diagnostic_data["Precio"] = y_train

(
    diagnostic_data
    .groupby("Version_Tier")["Precio"]
    .agg(
        count="count",
        median="median",
        mean="mean",
    )
    .sort_index()
)


In [ ]:
(
    diagnostic_data
    .groupby(["Marca", "Modelo", "Version_Tier"])["Precio"]
    .agg(
        count="count",
        median="median",
    )
    .reset_index()
    .sort_values(["Marca", "Modelo", "Version_Tier"])
)


In [ ]:
feature_importance = pd.Series(
    xgboost_single_model.named_steps["regressor"].feature_importances_,
    index=X_train_encoded.columns,
).sort_values(ascending=False)

feature_importance.head(30)


In [ ]:
feature_importance[feature_importance.index.str.contains("Version")]


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Error Analysis
</h3>


<h2 style="background-color: #d0ebff; color: #1a1a1a; display: inline-block; padding: 10px 18px; border-radius: 10px; font-size: 28px;">
Leakage-Safe Deduplication Experiments
</h2>

Esta sección concentra pruebas adicionales para evaluar si el desempeño mejora al tratar correctamente las filas repetidas. El objetivo es mantener un pipeline académico y evitar leakage:

- primero se eliminan duplicados exactos del dataset preprocesado;
- recién después se realiza un nuevo split train-validation;
- las categorías raras se aprenden solo con train y luego se aplican a validation;
- el one-hot encoding aprende sus columnas solo con train;
- cualquier transformación supervisada se calcula sin usar el target de validation.

Como el split cambia después de deduplicar, estos resultados deben compararse **dentro de esta sección** y no directamente contra métricas viejas obtenidas con otro split.

In [ ]:
import sys

import pandas as pd

sys.path.append("../src")

import experiment_utils as exp

TARGET = exp.TARGET
XGB_BASE_PARAMS = exp.XGB_BASE_PARAMS
XGB_REGULARIZED_PARAMS = exp.XGB_REGULARIZED_PARAMS


In [ ]:
dataset_pp = pd.read_csv("../data/dataset_pp.csv")
dataset_pp_dedup = exp.deduplicate_dataset(dataset_pp)

exp.build_deduplication_summary(dataset_pp, dataset_pp_dedup)


La deduplicación se realiza antes del split porque, si una misma publicación queda en train y validation, el modelo puede evaluarse sobre casos que ya vio. En ese escenario, la métrica de validation deja de representar correctamente la capacidad de generalización.

In [ ]:
X_train_raw, X_val_raw, y_train, y_val = exp.split_after_deduplication(dataset_pp_dedup)

exp.build_split_summary(X_train_raw, X_val_raw, y_train, y_val)


Primero se prueban variantes no supervisadas: cambios en columnas y agrupación de categorías raras. Estas variantes son simples de justificar porque no usan el target para crear features.

In [ ]:
one_hot_experiments = [
    {
        "experiment": "baseline_deduplicated",
        "feature_blocks": [],
        "rare_min_count": None,
        "drop_cols": [],
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
    {
        "experiment": "premium_feature",
        "feature_blocks": ["premium"],
        "rare_min_count": None,
        "drop_cols": [],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
    {
        "experiment": "premium_feature_without_color_and_camera",
        "feature_blocks": ["premium"],
        "rare_min_count": None,
        "drop_cols": ["Color", "Con cámara de retroceso"],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
    {
        "experiment": "premium_feature_rare20",
        "feature_blocks": ["premium"],
        "rare_min_count": 20,
        "drop_cols": [],
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
    {
        "experiment": "premium_and_brand_model_rare20",
        "feature_blocks": ["premium", "brand_model"],
        "rare_min_count": 20,
        "drop_cols": [],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
    {
        "experiment": "all_engineered_features_rare20",
        "feature_blocks": "all",
        "rare_min_count": 20,
        "drop_cols": [],
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
    {
        "experiment": "all_engineered_features_without_color_and_camera_rare20",
        "feature_blocks": "all",
        "rare_min_count": 20,
        "drop_cols": ["Color", "Con cámara de retroceso"],
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
]

one_hot_comparison, one_hot_fitted = exp.run_one_hot_experiments(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    experiments=one_hot_experiments,
)

exp.style_experiment_comparison(one_hot_comparison)


También se prueba una variante supervisada con target encoding de `Marca`, `Modelo` y `Marca_Modelo`. Para evitar leakage, el target encoding de train se calcula con folds internos: cada fila recibe un promedio aprendido sin usar su propio target. Validation se transforma usando únicamente estadísticas aprendidas en train.

In [ ]:
target_encoding_experiments = [
    {
        "experiment": "target_encoding_brand_model",
        "feature_blocks": ["brand_model"],
        "rare_min_count": 20,
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
    {
        "experiment": "target_encoding_brand_model_with_usage_and_premium",
        "feature_blocks": ["brand_model", "usage", "premium", "cilindrada_missing"],
        "rare_min_count": 20,
        "params_name": "base",
        "params": XGB_BASE_PARAMS,
    },
]

target_encoding_comparison, target_encoding_fitted = exp.run_target_encoding_experiments(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    experiments=target_encoding_experiments,
)

exp.style_experiment_comparison(target_encoding_comparison)


In [ ]:
experiment_comparison = (
    pd.concat([one_hot_comparison, target_encoding_comparison], ignore_index=True)
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

exp.style_experiment_comparison(experiment_comparison)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Label Encoding Experiment
</h3>

Además del one-hot encoding, se prueba una variante con **label encoding** para `Marca`, `Modelo` y `Marca_Modelo`. Esta prueba es exploratoria: estas variables son nominales, por lo que el número asignado a cada categoría no representa un orden real. Sin embargo, en modelos basados en árboles como XGBoost puede ser útil evaluarlo empíricamente, especialmente cuando hay muchas categorías y el one-hot genera una matriz muy amplia.

Para evitar leakage, el mapeo de categorías se aprende solamente con train. Las categorías de validation que no aparecen en train se codifican como `-1`.

In [ ]:
label_encoding_experiments = [
    {
        "experiment": "label_encoding_brand_model_only",
        "feature_blocks": ["brand_model", "premium"],
        "rare_min_count": None,
        "label_encoded_cols": ["Marca", "Modelo", "Marca_Modelo"],
        "drop_cols": ["Color", "Con cámara de retroceso"],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
    {
        "experiment": "label_encoding_brand_model_rare20",
        "feature_blocks": ["brand_model", "premium"],
        "rare_min_count": 20,
        "label_encoded_cols": ["Marca", "Modelo", "Marca_Modelo"],
        "drop_cols": ["Color", "Con cámara de retroceso"],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
    {
        "experiment": "label_encoding_all_categoricals",
        "feature_blocks": ["brand_model", "premium"],
        "rare_min_count": None,
        "label_encoded_cols": [
            "Marca", "Modelo", "Marca_Modelo", "Color",
            "Tipo de vendedor", "Tipo de combustible", "Transmisión",
        ],
        "drop_cols": [],
        "params_name": "regularized",
        "params": XGB_REGULARIZED_PARAMS,
    },
]

label_encoding_comparison, label_encoding_fitted = exp.run_label_encoding_experiments(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    experiments=label_encoding_experiments,
)

exp.style_experiment_comparison(label_encoding_comparison)


In [ ]:
experiment_comparison_with_label_encoding = (
    pd.concat(
        [experiment_comparison, label_encoding_comparison],
        ignore_index=True,
        sort=False,
    )
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

exp.style_experiment_comparison(experiment_comparison_with_label_encoding)


Si el label encoding no mejora respecto del one-hot, la interpretación es razonable: `Marca` y `Modelo` no tienen un orden natural, entonces el código numérico puede introducir una relación artificial entre categorías. Si mejora, debe reportarse como un resultado empírico específico de XGBoost y no como una codificación ordinal con significado propio.

Después de comparar todas las variantes, se selecciona el mejor experimento global según el RMSE de validation.

In [ ]:
best_experiment_name, best_experiment = exp.select_best_experiment(
    experiment_comparison_with_label_encoding,
    one_hot_fitted,
    target_encoding_fitted,
    label_encoding_fitted,
)

best_experiment_summary = exp.build_best_experiment_summary(
    experiment_comparison_with_label_encoding,
    best_experiment_name,
    best_experiment,
)

best_experiment_summary.style.hide(axis="index").format(exp.EXPERIMENT_METRIC_FORMAT)


Finalmente se inspeccionan los errores más grandes del mejor experimento. Esta tabla no se usa para ajustar el modelo automáticamente; sirve para diagnóstico manual de posibles outliers, errores de carga o segmentos donde faltan features relevantes.

In [ ]:
best_predictions = exp.attach_validation_context(
    best_experiment["predictions"],
    X_val_raw,
)

best_predictions[
    [
        "Marca", "Modelo", "Versión", "Año", "Kilómetros",
        "y_true", "y_pred", "residual", "abs_error",
    ]
].sort_values("abs_error", ascending=False).head(20).style.hide(axis="index").format({
    "y_true": "{:,.2f}",
    "y_pred": "{:,.2f}",
    "residual": "{:,.2f}",
    "abs_error": "{:,.2f}",
})
